# non-diff-fn-wrap — ex2: confirm sorted_computational_graph stops at a non-differentiable node

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `non-diff-fn-wrap`. Running the final beacon cell reports progress against the `Backprop: non-differentiable fn wrap` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: non-differentiable fn wrap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`non-diff-fn-wrap`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "non-diff-fn-wrap"
DD_SUBTOPIC = "Backprop: non-differentiable fn wrap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## non-differentiable wrap as a graph terminator — quick refresher

ex1 verified `is_differentiable=False` produces `requires_grad=False` and `recipe=None`. The deeper consequence: when `sorted_computational_graph` walks parents, it stops at any node with `recipe=None` — including the output of a non-diff op.

Concretely: if `mask = eq(a, b)` (non-diff), then `mask.recipe is None`, so even though `mask`'s downstream consumers may reference it, the reverse-pass walk treats `mask` as a leaf and does NOT recurse into `a` or `b` THROUGH `mask`.

**Why this matters.** Without the terminator behavior, the reverse pass would walk into a non-diff op (`torch.eq`), find no back_fn registered for it, and crash with `KeyError`. The `recipe=None` short-circuit is what makes detach/eq/argmax safe to use mid-graph.

### Exercise 2 — confirm sorted_computational_graph stops at a non-differentiable node

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the graph-termination consequence of recipe=None: a non-differentiable op's output behaves as a leaf during reverse-pass traversal, so its tensor parents are not reachable through it.
> Keywords: non-differentiable, graph-terminator, leaf, topo-walk
> ```

**KCs targeted:** `non-diff-fn-wrap`, `is-differentiable-flag`

ex1 verified the wrapper's local behavior: non-diff op → `requires_grad=False` and `recipe=None`. Here we exercise the DOWNSTREAM consequence: when `sorted_computational_graph` walks parents, it stops at any node with `recipe=None`.

You implement TWO things:

1. **`wrap_forward_fn(fwd_fn, is_differentiable=True)`** — same as ex1 (full wrapper that builds the Recipe conditionally). Provided in the stub as a guide; you finish the body.
2. **Build a 4-node compute graph by hand** and call `sorted_computational_graph(end_node)` (provided). Verify that the non-differentiable node terminates the walk.

We've provided `sorted_computational_graph` for you. You just have to build the graph and verify the structural property in the test body — that's the Analyze-level Bloom: you're examining the GRAPH SHAPE, not deriving math.

Setup cell provides `MiniTensor`, `Recipe`, `grad_tracking_enabled`. Don't call `torch.autograd`.

In [ ]:
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        # TODO: three-gate AND for requires_grad.
        # TODO: build MiniTensor; attach Recipe only when requires_grad.
        raise NotImplementedError()
    return tensor_func


def sorted_computational_graph(tensor):
    """Provided: reverse-topo sort (end node first). Stops at recipe=None."""
    result = []
    perm = set()
    def visit(cur):
        if id(cur) in perm:
            return
        perm.add(id(cur))
        if cur.recipe is not None:
            for p in cur.recipe.parents.values():
                visit(p)
        result.append(cur)
    visit(tensor)
    return result[::-1]


def _test_ex2():
    globals()['grad_tracking_enabled'] = True
    add = wrap_forward_fn(t.add)
    eq  = wrap_forward_fn(t.eq, is_differentiable=False)
    mul = wrap_forward_fn(t.multiply)

    # --- build:
    #   a, b are leaves
    #   mask = eq(a, b)       <-- non-diff: graph terminates here
    #   c    = add(a, b)
    #   d    = mul(c, c)      <-- end node
    a = MiniTensor(t.tensor([1.0, 2.0, 3.0]), requires_grad=True)
    b = MiniTensor(t.tensor([1.0, 0.0, 3.0]), requires_grad=True)
    mask = eq(a, b)
    c = add(a, b)
    d = mul(c, c)

    # --- structural invariants on the WRAPPED outputs ---
    assert mask.recipe is None, 'eq output must have recipe=None (graph terminator)'
    assert mask.requires_grad is False
    assert c.recipe is not None and c.recipe.func is t.add
    assert d.recipe is not None and d.recipe.func is t.multiply

    # --- walk from d: only d, c, a, b reachable; mask is NOT in the graph ---
    graph = sorted_computational_graph(d)
    ids = {id(n) for n in graph}
    assert id(d) in ids and id(c) in ids and id(a) in ids and id(b) in ids
    assert id(mask) not in ids, (
        'mask not consumed by d — should not appear; this is a sanity check'
    )

    # --- walk from mask: traversal STOPS at mask (it is itself a leaf) ---
    mask_graph = sorted_computational_graph(mask)
    mask_ids = {id(n) for n in mask_graph}
    # mask has recipe=None → topo walk treats it as a leaf, so only `mask` appears.
    assert mask_ids == {id(mask)}, (
        f'non-diff output should be a graph leaf, but walk found {mask_ids} '
        f'(expected {{id(mask)}}) — recipe-None termination broken'
    )
    assert id(a) not in mask_ids and id(b) not in mask_ids, (
        'parents of non-diff op MUST NOT be reachable through it'
    )

    # --- now build a graph that CONSUMES mask — the same termination still holds ---
    # (We can multiply tensors elementwise by a bool tensor — torch promotes.)
    weighted = mul(c, mask)   # c * mask. Note mask.requires_grad=False → weighted...
    # weighted.requires_grad is True (c is tracked), but its parents dict only contains
    # MiniTensor inputs. The walk reaches mask via parents and STOPS there.
    weighted_graph = sorted_computational_graph(weighted)
    w_ids = {id(n) for n in weighted_graph}
    assert id(weighted) in w_ids and id(c) in w_ids and id(mask) in w_ids
    # Crucially: mask's parents (a, b through eq) are NOT reachable via mask.
    # But a and b ARE reachable via c. So they appear — just not THROUGH mask.
    # Verify by checking the position invariant.
    pos = {id(n): i for i, n in enumerate(weighted_graph)}
    # mask comes after weighted (since weighted's recipe lists mask as parent).
    assert pos[id(weighted)] < pos[id(mask)], 'parent before child in reverse order'
    # But mask is itself a leaf in the walk — its 'parents' (a, b through eq) are
    # not traversed THROUGH mask. They're traversed THROUGH c instead.
    # We assert by replacing c temporarily so mask is the only path to a/b:
    # easier: just verify the walk would have crashed without termination by
    # checking that mask's recipe is None.
    assert mask.recipe is None, 'final invariant: mask terminates the walk'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            globals()['grad_tracking_enabled']
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

**Why graph termination is necessary, not optional.** Without it, `sorted_computational_graph` recursing into `mask.recipe.parents` would walk into `(a, b)` via the eq op, then the reverse pass would look up `(t.eq, 0)` in `BACK_FUNCS` — `KeyError`. The `recipe is None` short-circuit is what makes detach / eq / argmax safe to use mid-graph.

**Why this is an Analyze-level exercise.** You're not computing gradients or implementing math — you're examining the GRAPH SHAPE to confirm a structural invariant holds. The cognitive operation is verification of a system-level property: 'does this design decision (recipe=None for non-diff) prevent the failure mode it was supposed to prevent?'

**Why mask's parents are still reachable via c.** Each downstream tensor knows its own parents; multiple paths to the same leaf is the diamond-DAG pattern. The point is that they're not reachable THROUGH the non-differentiable node. Without termination, the walk would erroneously go through mask AND through c, and the dispatcher would crash on the eq lookup.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()